In [ ]:
df = pd.read_csv('energy.csv')
print(df.columns)

In [84]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.graph_objects as go

# 1. LOAD AND AGGREGATE DATA
df = pd.read_csv('energy.csv')

# Aggregate specific fuel columns into the categories your project specified
fossil_keywords = ['.Coal', '.Natural Gas', '.Petroleum', '.Distillate Fuel Oil', '.Kerosene']
renew_keywords = ['.Solar', '.Wind', '.Wood', '.Geothermal', '.Hydropower']

consumption_cols = [c for c in df.columns if 'Consumption.' in c]
fossil_cols = [c for c in consumption_cols if any(k in c for k in fossil_keywords)]
renew_cols = [c for c in consumption_cols if any(k in c for k in renew_keywords)]

df['Fossil_Total'] = df[fossil_cols].sum(axis=1)
df['Renew_Total'] = df[renew_cols].sum(axis=1)
df['Total_Energy'] = df[consumption_cols].sum(axis=1)
df['Renew_Share'] = (df['Renew_Total'] / df['Total_Energy']) * 100

# 2. INITIALIZE DASHBOARD
app = dash.Dash(__name__)

app.layout = html.Div(style={'fontFamily': 'Arial', 'padding': '20px'}, children=[
    html.H1("US Energy Transition Explorer"),
    html.Div([
        html.Label("Select State: "),
        dcc.Dropdown(id='state-dropdown', options=[{'label': s, 'value': s} for s in sorted(df['State'].unique())], value='California')
    ], style={'width': '300px', 'marginBottom': '20px'}),

    html.Div(id='policy-verdict', style={'padding': '15px', 'fontSize': '20px', 'borderRadius': '5px', 'textAlign': 'center', 'marginBottom': '20px'}),

    html.Div(style={'display': 'flex'}, children=[
        dcc.Graph(id='abs-chart', style={'width': '50%'}),
        dcc.Graph(id='share-chart', style={'width': '50%'})
    ])
])

@app.callback(
    [Output('abs-chart', 'figure'), Output('share-chart', 'figure'), Output('policy-verdict', 'children'), Output('policy-verdict', 'style')],
    [Input('state-dropdown', 'value')]
)
def update(state):
    dff = df[df['State'] == state].sort_values('Year')
    
    # Logic for Step 6: Deploy / Automated Verdict
    f_start, f_end = dff['Fossil_Total'].iloc[0], dff['Fossil_Total'].iloc[-1]
    r_start, r_end = dff['Renew_Total'].iloc[0], dff['Renew_Total'].iloc[-1]
    
    if f_end < f_start and r_end > r_start:
        verdict = "VERDICT: TRUE TRANSITION (Fossils are being displaced)"
        style = {'backgroundColor': '#e6ffed', 'color': '#22863a', 'border': '1px solid #22863a'}
    else:
        verdict = "VERDICT: ADDITIVE GROWTH (Renewables added on top of fossil use)"
        style = {'backgroundColor': '#fffdef', 'color': '#735c0f', 'border': '1px solid #d4a017'}

    # Task 1 Chart
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Fossil_Total'], name="Fossils", line=dict(color='#D85A30', width=3)))
    fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Total'], name="Renewables", line=dict(color='#639922', width=3)))
    fig1.update_layout(title="Task 1: Absolute Consumption", xaxis_title="Year", yaxis_title="BTU Equivalents")

    # Task 2 Chart
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Share'], fill='tozeroy', name="Renewable %", line=dict(color='#639922')))
    fig2.update_layout(title="Task 2: Proportional Energy Mix (%)", xaxis_title="Year", yaxis_title="Percent Share")

    return fig1, fig2, verdict, style

if __name__ == '__main__':
    app.run(debug=True)

In [96]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. LOAD AND PREP DATA
df = pd.read_csv('energy.csv')

# Mapping for Map visual
state_to_code = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO',
    'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
    'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC'
}
df['StateCode'] = df['State'].map(state_to_code)

# Aggregate categories
fossil_keywords = ['.Coal', '.Natural Gas', '.Petroleum', '.Distillate Fuel Oil', '.Kerosene']
renew_keywords = ['.Solar', '.Wind', '.Wood', '.Geothermal', '.Hydropower']
consumption_cols = [c for c in df.columns if 'Consumption.' in c]
fossil_cols = [c for c in consumption_cols if any(k in c for k in fossil_keywords)]
renew_cols = [c for c in consumption_cols if any(k in c for k in renew_keywords)]

df['Fossil_Total'] = df[fossil_cols].sum(axis=1)
df['Renew_Total'] = df[renew_cols].sum(axis=1)
df['Total_Energy'] = df[consumption_cols].sum(axis=1)
df['Renew_Share'] = (df['Renew_Total'] / df['Total_Energy']) * 100

# Latest year for the map coloring
latest_year = df['Year'].max()
df_map = df[df['Year'] == latest_year]

# 2. INITIALIZE DASHBOARD
app = dash.Dash(__name__)

app.layout = html.Div(style={'fontFamily': 'Arial', 'padding': '20px', 'backgroundColor': '#f4f4f4'}, children=[
    html.H1("US Energy Transition: Policy Overview Map", style={'textAlign': 'center'}),
    html.P("Click a state on the map to investigate its transition status.", style={'textAlign': 'center'}),

    # MAP VIEW (View A)
    html.Div([
        dcc.Graph(
            id='usa-map',
            figure=px.choropleth(
                df_map,
                locations='StateCode',
                locationmode="USA-states",
                color='Renew_Share',
                scope="usa",
                color_continuous_scale="Viridis",
                labels={'Renew_Share': 'Renewable %'},
                title=f"Renewable Portfolio Share by State ({latest_year})"
            ).update_layout(clickmode='event+select', margin={"r":0,"t":40,"l":0,"b":0})
        )
    ], style={'backgroundColor': 'white', 'padding': '10px', 'borderRadius': '10px', 'marginBottom': '20px'}),

    # VERDICT & GRAPHS (View B & C)
    html.Div(id='policy-verdict', style={'padding': '15px', 'fontSize': '22px', 'borderRadius': '5px', 'textAlign': 'center', 'marginBottom': '20px', 'fontWeight': 'bold'}),

    html.Div(style={'display': 'flex', 'gap': '20px'}, children=[
        dcc.Graph(id='abs-chart', style={'width': '50%', 'backgroundColor': 'white', 'borderRadius': '10px'}),
        dcc.Graph(id='share-chart', style={'width': '50%', 'backgroundColor': 'white', 'borderRadius': '10px'})
    ])
])

# 3. INTERACTIVITY CALLBACK (Map Click -> Update Graphs)
@app.callback(
    [Output('abs-chart', 'figure'), 
     Output('share-chart', 'figure'), 
     Output('policy-verdict', 'children'), 
     Output('policy-verdict', 'style')],
    [Input('usa-map', 'clickData')]
)
def update_from_map(clickData):
    # Default to California if nothing is clicked
    state_name = "California"
    if clickData:
        state_name = clickData['points'][0]['location']
        # Convert code back to name if necessary
        code_to_state = {v: k for k, v in state_to_code.items()}
        state_name = code_to_state.get(state_name, "California")

    dff = df[df['State'] == state_name].sort_values('Year')
    
    # Automated Verdict Logic
    f_start, f_end = dff['Fossil_Total'].iloc[0], dff['Fossil_Total'].iloc[-1]
    r_start, r_end = dff['Renew_Total'].iloc[0], dff['Renew_Total'].iloc[-1]
    
    if f_end < f_start and r_end > r_start:
        verdict = f"{state_name} Verdict: TRUE TRANSITION (Fossils Decreasing)"
        style = {'backgroundColor': '#e6ffed', 'color': '#22863a', 'border': '2px solid #22863a'}
    else:
        verdict = f"{state_name} Verdict: ADDITIVE GROWTH (Renewables added on top)"
        style = {'backgroundColor': '#fffdef', 'color': '#735c0f', 'border': '2px solid #d4a017'}

    # Task 1: Absolute
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Fossil_Total'], name="Fossils", line=dict(color='#D85A30', width=3)))
    fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Total'], name="Renewables", line=dict(color='#639922', width=3)))
    fig1.update_layout(title=f"Task 1: Absolute Trends - {state_name}", xaxis_title="Year", yaxis_title="BTU")

    # Task 2: Share
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Share'], fill='tozeroy', name="Renewable %", line=dict(color='#639922')))
    fig2.update_layout(title=f"Task 2: Percentage of Renewable Energy (%) - {state_name}", xaxis_title="Year", yaxis_title="Percent")

    # NEW Chart for Task 3: Fossil Breakdown
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Consumption.Coal'], name="Coal", stackgroup='one'))
    fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Consumption.Natural Gas'], name="Natural Gas", stackgroup='one'))
    fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Consumption.Petroleum'], name="Petroleum", stackgroup='one'))
    fig3.update_layout(title="Task 3: Fossil Fuel Composition (Bridge Fuel Analysis)")

    return fig1, fig2, fig3, verdict, style

if __name__ == '__main__':
    app.run(debug=True)

[2026-04-09 11:41:40,008] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "/Users/nandi/Documents/coding/AI-Usage-In-Departmental-Acceptances-at-UNC-Chapel-Hill/.conda/lib/python3.11/site-packages/pandas/core/indexes/base.py", line 3812, in get_loc
    return self._engine.get_loc(casted_key)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pandas/_libs/index.pyx", line 167, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/index.pyx", line 196, in pandas._libs.index.IndexEngine.get_loc
  File "pandas/_libs/hashtable_class_helper.pxi", line 7088, in pandas._libs.hashtable.PyObjectHashTable.get_item
  File "pandas/_libs/hashtable_class_helper.pxi", line 7096, in pandas._libs.hashtable.PyObjectHashTable.get_item
KeyError: 'Consumption.Coal'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/nandi/Documents/coding/AI-Usage-In-Departmental-Acceptances-at-

In [94]:
# import dash
# from dash import dcc, html, Input, Output
# import pandas as pd
# import plotly.express as px
# import plotly.graph_objects as go

# # 1. DATA PREPARATION & AGGREGATION
# df = pd.read_csv('energy.csv')

# # Mapping for Map visual
# state_to_code = {
#     'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
#     'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
#     'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
#     'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
#     'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO',
#     'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
#     'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
#     'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
#     'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
#     'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
#     'District of Columbia': 'DC'
# }
# df['StateCode'] = df['State'].map(state_to_code)

# # Pre-aggregate specific fuel types across all sectors (Task 3 requirement)
# cols = df.columns
# df['Total_Coal'] = df[[c for c in cols if 'Consumption.' in c and '.Coal' in c]].sum(axis=1)
# df['Total_Gas'] = df[[c for c in cols if 'Consumption.' in c and '.Natural Gas' in c]].sum(axis=1)
# petro_keys = ['.Petroleum', '.Distillate Fuel Oil', '.Kerosene', '.Other Petroleum Products']
# df['Total_Petro'] = df[[c for c in cols if 'Consumption.' in c and any(k in c for k in petro_keys)]].sum(axis=1)

# # Renewables and Totals (Task 1 & 2)
# renew_keys = ['.Solar', '.Wind', '.Wood', '.Geothermal', '.Hydropower']
# df['Renew_Total'] = df[[c for c in cols if 'Consumption.' in c and any(k in c for k in renew_keys)]].sum(axis=1)
# df['Fossil_Total'] = df['Total_Coal'] + df['Total_Gas'] + df['Total_Petro']
# df['Total_Energy'] = df[[c for c in cols if 'Consumption.' in c]].sum(axis=1)
# df['Renew_Share'] = (df['Renew_Total'] / df['Total_Energy']) * 100

# # Latest year for initial map view
# df_map = df[df['Year'] == df['Year'].max()]

# # 2. DASHBOARD LAYOUT
# app = dash.Dash(__name__)

# app.layout = html.Div(style={'fontFamily': 'Arial', 'padding': '20px', 'backgroundColor': '#f2f2f2'}, children=[
#     html.H1("Energy Transition Study: Iterative Policy Dashboard", style={'textAlign': 'center'}),
    
#     # ROW 1: THE MAP (Selection tool)
#     html.Div([
#         dcc.Graph(
#             id='usa-map',
#             figure=px.choropleth(
#                 df_map, locations='StateCode', locationmode="USA-states",
#                 color='Renew_Share', scope="usa", color_continuous_scale="Greens",
#                 title="Select a State to View Detailed Analysis"
#             ).update_layout(margin={"r":0,"t":40,"l":0,"b":0})
#         )
#     ], style={'backgroundColor': 'white', 'padding': '10px', 'borderRadius': '15px', 'marginBottom': '20px'}),

#     # ROW 2: THE AUTOMATED VERDICT
#     html.Div(id='verdict-box', style={'padding': '20px', 'fontSize': '22px', 'textAlign': 'center', 'borderRadius': '10px', 'marginBottom': '20px', 'fontWeight': 'bold'}),

#     # ROW 3: THREE CO-ORDINATED CHARTS
#     html.Div(style={'display': 'flex', 'gap': '15px'}, children=[
#         html.Div([dcc.Graph(id='task1-abs')], style={'width': '33%', 'backgroundColor': 'white', 'borderRadius': '10px'}),
#         html.Div([dcc.Graph(id='task2-share')], style={'width': '33%', 'backgroundColor': 'white', 'borderRadius': '10px'}),
#         html.Div([dcc.Graph(id='task3-breakdown')], style={'width': '33%', 'backgroundColor': 'white', 'borderRadius': '10px'})
#     ])
# ])

# # 3. INTERACTIVITY CALLBACK
# @app.callback(
#     [Output('task1-abs', 'figure'), Output('task2-share', 'figure'), 
#      Output('task3-breakdown', 'figure'), Output('verdict-box', 'children'), 
#      Output('verdict-box', 'style')],
#     [Input('usa-map', 'clickData')]
# )
# def update_dashboard(clickData):
#     # Default State
#     state_code = clickData['points'][0]['location'] if clickData else "NY"
#     code_to_state = {v: k for k, v in state_to_code.items()}
#     state_name = code_to_state.get(state_code, "New York")
    
#     # Filter data for selected state
#     dff = df[df['State'] == state_name].sort_values('Year')
    
#     # Logic for Verdict
#     f_change = dff['Fossil_Total'].iloc[-1] - dff['Fossil_Total'].iloc[0]
#     r_change = dff['Renew_Total'].iloc[-1] - dff['Renew_Total'].iloc[0]
    
#     if f_change < 0 and r_change > 0:
#         verdict = f"{state_name}: TRUE TRANSITION (Fossils Decreasing)"
#         v_style = {'backgroundColor': '#d4edda', 'color': '#155724', 'border': '2px solid #c3e6cb'}
#     else:
#         verdict = f"{state_name}: ADDITIVE GROWTH (Renewables added to Fossil baseline)"
#         v_style = {'backgroundColor': '#fff3cd', 'color': '#856404', 'border': '2px solid #ffeeba'}

#     # TASK 1: Absolute Comparison
#     fig1 = go.Figure()
#     fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Fossil_Total'], name="Fossils", line=dict(color='#D85A30', width=3)))
#     fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Total'], name="Renewables", line=dict(color='#28a745', width=3)))
#     fig1.update_layout(title="Task 1: Absolute Trends", xaxis_title="Year", yaxis_title="BTU")

#     # TASK 2: Proportional Share
#     fig2 = go.Figure()
#     fig2.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Share'], fill='tozeroy', name="Renewable %", line=dict(color='#28a745')))
#     fig2.update_layout(title="Task 2: Market Share (%)", xaxis_title="Year", yaxis_title="Percentage")

#     # TASK 3: Fossil Breakdown (The Stakeholder Request)
#     fig3 = go.Figure()
#     fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Total_Coal'], name="Coal", stackgroup='one', line=dict(color='#333333')))
#     fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Total_Gas'], name="Natural Gas", stackgroup='one', line=dict(color='#5bc0de')))
#     fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Total_Petro'], name="Petroleum", stackgroup='one', line=dict(color='#f0ad4e')))
#     fig3.update_layout(title="Task 3: Fossil Composition (Policy Detail)", xaxis_title="Year", yaxis_title="BTU")

#     return fig1, fig2, fig3, verdict, v_style

# if __name__ == '__main__':
#     app.run(debug=True)

In [98]:
import dash
from dash import dcc, html, Input, Output
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# 1. DATA PREPARATION
df = pd.read_csv('energy.csv')

state_to_code = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'Florida': 'FL', 'Georgia': 'GA',
    'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN', 'Iowa': 'IA',
    'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME', 'Maryland': 'MD',
    'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS', 'Missouri': 'MO',
    'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH', 'New Jersey': 'NJ',
    'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND', 'Ohio': 'OH',
    'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI', 'South Carolina': 'SC',
    'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT', 'Vermont': 'VT',
    'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY',
    'District of Columbia': 'DC'
}
df['StateCode'] = df['State'].map(state_to_code)

# Aggregation
cols = df.columns
df['Total_Coal'] = df[[c for c in cols if 'Consumption.' in c and '.Coal' in c]].sum(axis=1)
df['Total_Gas'] = df[[c for c in cols if 'Consumption.' in c and '.Natural Gas' in c]].sum(axis=1)
petro_keys = ['.Petroleum', '.Distillate Fuel Oil', '.Kerosene', '.Other Petroleum Products']
df['Total_Petro'] = df[[c for c in cols if 'Consumption.' in c and any(k in c for k in petro_keys)]].sum(axis=1)
renew_keys = ['.Solar', '.Wind', '.Wood', '.Geothermal', '.Hydropower']
df['Renew_Total'] = df[[c for c in cols if 'Consumption.' in c and any(k in c for k in renew_keys)]].sum(axis=1)
df['Fossil_Total'] = df['Total_Coal'] + df['Total_Gas'] + df['Total_Petro']
df['Total_Energy'] = df[[c for c in cols if 'Consumption.' in c]].sum(axis=1)
df['Renew_Share'] = (df['Renew_Total'] / df['Total_Energy']) * 100

df_map = df[df['Year'] == df['Year'].max()]

# 2. DASHBOARD LAYOUT
app = dash.Dash(__name__)

app.layout = html.Div(style={'fontFamily': 'Times New Roman', 'padding': '40px', 'backgroundColor': '#f8f9fa'}, children=[
    html.H1("US Energy Transition: Strategic Policy Tool", style={'textAlign': 'center', 'marginBottom': '30px'}),
    
    # MAP - Now taller
    html.Div([
        dcc.Graph(
            id='usa-map',
            style={'height': '600px'}, # INCREASED HEIGHT
            figure=px.choropleth(
                df_map, locations='StateCode', locationmode="USA-states",
                color='Renew_Share', scope="usa", color_continuous_scale="YlGn",
                title="State Renewable Share Overview (Click a State)"
            ).update_layout(margin={"r":0,"t":50,"l":0,"b":0})
        )
    ], style={'backgroundColor': 'white', 'padding': '20px', 'borderRadius': '15px', 'boxShadow': '0px 4px 12px rgba(0,0,0,0.1)', 'marginBottom': '30px'}),

    # VERDICT BANNER
    html.Div(id='verdict-box', style={'padding': '25px', 'fontSize': '26px', 'textAlign': 'center', 'borderRadius': '12px', 'marginBottom': '30px', 'fontWeight': 'bold'}),

    # ROW 1: Task 1 and Task 2 (Side by Side)
    html.Div(style={'display': 'flex', 'gap': '25px', 'marginBottom': '30px'}, children=[
        html.Div([
            dcc.Graph(id='task1-abs', style={'height': '500px'}) # INCREASED HEIGHT
        ], style={'width': '50%', 'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '15px', 'boxShadow': '0px 4px 12px rgba(0,0,0,0.1)'}),
        
        html.Div([
            dcc.Graph(id='task2-share', style={'height': '500px'}) # INCREASED HEIGHT
        ], style={'width': '50%', 'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '15px', 'boxShadow': '0px 4px 12px rgba(0,0,0,0.1)'})
    ]),

    # ROW 2: Task 3 (Full Width Breakdown)
    html.Div([
        dcc.Graph(id='task3-breakdown', style={'height': '550px'}) # WIDE AND TALL
    ], style={'width': '100%', 'backgroundColor': 'white', 'padding': '15px', 'borderRadius': '15px', 'boxShadow': '0px 4px 12px rgba(0,0,0,0.1)'})
])

# 3. CALLBACK
@app.callback(
    [Output('task1-abs', 'figure'), Output('task2-share', 'figure'), 
     Output('task3-breakdown', 'figure'), Output('verdict-box', 'children'), 
     Output('verdict-box', 'style')],
    [Input('usa-map', 'clickData')]
)
def update_dashboard(clickData):
    state_code = clickData['points'][0]['location'] if clickData else "NY"
    code_to_state = {v: k for k, v in state_to_code.items()}
    state_name = code_to_state.get(state_code, "New York")
    dff = df[df['State'] == state_name].sort_values('Year')
    
    # Verdict Logic
    f_change = dff['Fossil_Total'].iloc[-1] - dff['Fossil_Total'].iloc[0]
    r_change = dff['Renew_Total'].iloc[-1] - dff['Renew_Total'].iloc[0]
    if f_change < 0 and r_change > 0:
        verdict = f"{state_name}: TRUE TRANSITION"
        v_style = {'backgroundColor': '#28a745', 'color': 'white'}
    else:
        verdict = f"{state_name}: ADDITIVE GROWTH"
        v_style = {'backgroundColor': '#ffc107', 'color': '#212529'}

    # Fig 1
    fig1 = go.Figure()
    fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Fossil_Total'], name="Fossils", line=dict(color='#D85A30', width=4)))
    fig1.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Total'], name="Renewables", line=dict(color='#28a745', width=4)))
    fig1.update_layout(title=f"Task 1: Absolute Consumption (BTU) - {state_name}", template='plotly_white')

    # Fig 2
    fig2 = go.Figure()
    fig2.add_trace(go.Scatter(x=dff['Year'], y=dff['Renew_Share'], fill='tozeroy', name="Renewable %", line=dict(color='#28a745', width=3)))
    fig2.update_layout(title = f"Task 2: Market Share (%) - {state_name}", template='plotly_white')

    # Fig 3 - Stacked Area
    fig3 = go.Figure()
    fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Total_Coal'], name="Coal", stackgroup='fossil', line=dict(color='#343a40')))
    fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Total_Gas'], name="Natural Gas", stackgroup='fossil', line=dict(color='#17a2b8')))
    fig3.add_trace(go.Scatter(x=dff['Year'], y=dff['Total_Petro'], name="Petroleum", stackgroup='fossil', line=dict(color='#fd7e14')))
    fig3.update_layout(title=f"Task 3: Detailed Fossil Breakdown - {state_name}", template='plotly_white')

    return fig1, fig2, fig3, verdict, v_style

if __name__ == '__main__':
    app.run(debug=True)